# 03 — Transfer Learning (ResNet50 fine-tuned)

Loads ImageNet-pretrained ResNet50, replaces the head, and fine-tunes on the fundus dataset in two stages:
1. **Warm-up**: freeze the backbone, train only the new classification head.
2. **Fine-tune**: unfreeze the top blocks of the backbone and train end-to-end at a low LR.

Swap `BACKBONE = "resnet50"` for `"vgg16"` below if you'd rather compare against VGG — the rest of the notebook is unchanged.

In [ ]:
import os, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50, VGG16
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_preprocess
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

BACKBONE = "resnet50"   # "resnet50" or "vgg16"

CONFIG = {
    "DATA_DIR": "/kaggle/working/dataset_extracted/splits",
    "IMG_DIR": "/kaggle/working/dataset_extracted/images",
    "LABEL_COL": "unified_label",
    "FILENAME_COL": "filename",
    "IMG_SIZE": (224, 224),
    "BATCH_SIZE": 32,
    "WARMUP_EPOCHS": 8,
    "FINETUNE_EPOCHS": 20,
    "SEED": 42,
}

CLASSES = ["Normal", "Diabetic Retinopathy", "Others", "Glaucoma",
           "Cataract", "Myopia", "AMD", "Hypertension"]

os.makedirs("artifacts", exist_ok=True)
tf.random.set_seed(CONFIG["SEED"])
preprocess_fn = resnet_preprocess if BACKBONE == "resnet50" else vgg_preprocess

In [ ]:
train_df = pd.read_csv(os.path.join(CONFIG["DATA_DIR"], "train.csv"))
val_df   = pd.read_csv(os.path.join(CONFIG["DATA_DIR"], "val.csv"))
test_df  = pd.read_csv(os.path.join(CONFIG["DATA_DIR"], "test.csv"))
for df in (train_df, val_df, test_df):
    df[CONFIG["LABEL_COL"]] = df[CONFIG["LABEL_COL"]].astype(str)

In [ ]:
train_aug = ImageDataGenerator(
    preprocessing_function=preprocess_fn,
    rotation_range=15,
    width_shift_range=0.08,
    height_shift_range=0.08,
    zoom_range=0.1,
    horizontal_flip=True,
)
eval_aug = ImageDataGenerator(preprocessing_function=preprocess_fn)

def make_gen(datagen, df, shuffle):
    return datagen.flow_from_dataframe(
        df, directory=CONFIG["IMG_DIR"],
        x_col=CONFIG["FILENAME_COL"], y_col=CONFIG["LABEL_COL"],
        target_size=CONFIG["IMG_SIZE"], batch_size=CONFIG["BATCH_SIZE"],
        class_mode="categorical", classes=CLASSES, shuffle=shuffle, seed=CONFIG["SEED"],
    )

train_gen = make_gen(train_aug, train_df, shuffle=True)
val_gen   = make_gen(eval_aug, val_df, shuffle=False)
test_gen  = make_gen(eval_aug, test_df, shuffle=False)

y_train_idx = train_gen.classes
weights = compute_class_weight("balanced", classes=np.unique(y_train_idx), y=y_train_idx)
class_weight_dict = dict(zip(np.unique(y_train_idx), weights))

## Build model: pretrained backbone + new head

In [ ]:
def build_backbone(name, input_shape):
    if name == "resnet50":
        return ResNet50(include_top=False, weights="imagenet", input_shape=input_shape, pooling="avg")
    elif name == "vgg16":
        return VGG16(include_top=False, weights="imagenet", input_shape=input_shape, pooling="avg")
    raise ValueError(name)

input_shape = CONFIG["IMG_SIZE"] + (3,)
backbone = build_backbone(BACKBONE, input_shape)
backbone.trainable = False   # stage 1: frozen

inputs = layers.Input(shape=input_shape)
x = backbone(inputs, training=False)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(len(CLASSES), activation="softmax")(x)
model = models.Model(inputs, outputs, name=f"{BACKBONE}_finetuned")

model.compile(optimizer=optimizers.Adam(1e-3), loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

## Stage 1 — train the new head only (backbone frozen)

In [ ]:
cbs_warmup = [
    callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
]

history_warmup = model.fit(
    train_gen, validation_data=val_gen,
    epochs=CONFIG["WARMUP_EPOCHS"],
    class_weight=class_weight_dict,
    callbacks=cbs_warmup,
)

## Stage 2 — unfreeze top of backbone, fine-tune at low LR

In [ ]:
backbone.trainable = True
# Keep the earliest layers frozen (generic features); fine-tune only the top ~30%
n_layers = len(backbone.layers)
freeze_until = int(n_layers * 0.7)
for layer in backbone.layers[:freeze_until]:
    layer.trainable = False

model.compile(optimizer=optimizers.Adam(1e-5), loss="categorical_crossentropy", metrics=["accuracy"])

cbs_ft = [
    callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7),
    callbacks.ModelCheckpoint(f"artifacts/{BACKBONE}_finetuned_best.keras", monitor="val_loss", save_best_only=True),
]

history_ft = model.fit(
    train_gen, validation_data=val_gen,
    epochs=CONFIG["FINETUNE_EPOCHS"],
    class_weight=class_weight_dict,
    callbacks=cbs_ft,
)

In [ ]:
def concat_history(h1, h2, key):
    return h1.history[key] + h2.history[key]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(concat_history(history_warmup, history_ft, "loss"), label="train")
axes[0].plot(concat_history(history_warmup, history_ft, "val_loss"), label="val")
axes[0].axvline(CONFIG["WARMUP_EPOCHS"], color="gray", ls="--", label="unfreeze")
axes[0].set_title("Loss"); axes[0].legend()

axes[1].plot(concat_history(history_warmup, history_ft, "accuracy"), label="train")
axes[1].plot(concat_history(history_warmup, history_ft, "val_accuracy"), label="val")
axes[1].axvline(CONFIG["WARMUP_EPOCHS"], color="gray", ls="--", label="unfreeze")
axes[1].set_title("Accuracy"); axes[1].legend()
plt.tight_layout()
plt.savefig(f"artifacts/{BACKBONE}_training_curves.png", dpi=150)
plt.show()

## Evaluate — per-class precision / recall / F1 (required deliverable)

In [ ]:
test_gen.reset()
y_true = test_gen.classes
y_prob = model.predict(test_gen, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

report = classification_report(y_true, y_pred, target_names=CLASSES, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report).T
print(classification_report(y_true, y_pred, target_names=CLASSES, zero_division=0))
report_df.to_csv(f"artifacts/{BACKBONE}_classification_report.csv")
report_df

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES)
plt.title(f"{BACKBONE} (fine-tuned) — Confusion Matrix (Test Set)")
plt.ylabel("True"); plt.xlabel("Predicted")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig(f"artifacts/{BACKBONE}_confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
n_bench = 50
sample_batch = next(iter(test_gen))[0][:min(n_bench, CONFIG["BATCH_SIZE"])]
model.predict(sample_batch, verbose=0)  # warmup

start = time.time()
for _ in range(10):
    model.predict(sample_batch, verbose=0)
elapsed = (time.time() - start) / 10
ms_per_image = elapsed / len(sample_batch) * 1000
print(f"{BACKBONE}: {ms_per_image:.2f} ms/image, {1000/ms_per_image:.1f} FPS")

metrics_summary = {
    "model": f"{BACKBONE}_finetuned",
    "test_accuracy": float(report["accuracy"]),
    "macro_f1": float(report["macro avg"]["f1-score"]),
    "weighted_f1": float(report["weighted avg"]["f1-score"]),
    "ms_per_image": ms_per_image,
    "n_params": int(model.count_params()),
}
with open(f"artifacts/{BACKBONE}_metrics.json", "w") as f:
    json.dump(metrics_summary, f, indent=2)
metrics_summary

Model, curves, report, confusion matrix and timing are saved under `artifacts/`. Next: `04_model_comparison.ipynb`.